# Radiation trapping

An optically thick line is re-absorbed before it leaves the plasma; only a fraction, the escape factor $g$, of its spontaneous emission gets out. OpenCR applies escape factors explicitly: given per line in `Conditions.escape`, or computed self-consistently from the populations with `Conditions.trapping` (the formulas are in [The CR model](/OpenCR/guides/model/)).

In [ ]:
import numpy as np
import plasma_plots

import opencr
from opencr.kinetics.radiation import escape_factor

model = opencr.load_model("hydrogen/atomic/basic", n_max=12)

## The approximations

Mewe's interpolation and Holstein's asymptotic forms against the line-centre optical depth $\tau_0$:

In [ ]:
tau = np.geomspace(1e-2, 1e6, 200)
with plasma_plots.figure(1, 1) as fig:
    for name in ("mewe", "holstein-slab", "holstein-cylinder"):
        fig[0].loglog(tau, escape_factor(tau, name), label=name)
    fig[0].set_xlabel(r"$\tau_0$")
    fig[0].set_ylabel("escape factor")
    fig[0].set_title("escape factor against optical depth")
    fig[0].legend()

## A self-consistent solve

The Lyman lines are absorbed by the ground state, whose density is prescribed here, and the Balmer lines by n=2, whose density is part of the solution; the solve iterates the escape factors and the populations to self-consistency. The absorption length `length` (the distance to the wall) is the parameter of the approximation:

In [ ]:
base = opencr.Conditions(3.0, 1e18, densities={"H(n=1)": 1e19, "H+": 1e18})
thin = model.solve_steady_state(base)
lengths = np.geomspace(1e-4, 10.0, 25)
n2, n3, g21, g32 = [], [], [], []
for length in lengths:
    thick = model.solve_steady_state(
        base.replace(trapping=opencr.RadiationTrapping(length))
    )
    n2.append(thick.population("H(n=2)") / thin.population("H(n=2)"))
    n3.append(thick.population("H(n=3)") / thin.population("H(n=3)"))
    g21.append(thick.conditions.escape["A:2-1"])
    g32.append(thick.conditions.escape["A:3-2"])
print(thick.report.summary())

In [ ]:
with plasma_plots.figure(1, 2) as fig:
    fig[0].loglog(lengths, n2, label="H(n=2)")
    fig[0].loglog(lengths, n3, label="H(n=3)")
    fig[0].set_xlabel("absorption length (m)")
    fig[0].set_ylabel("population / optically thin")
    fig[0].set_title("excited populations")
    fig[0].legend()
    fig[1].loglog(lengths, g21, label="Ly-α")
    fig[1].loglog(lengths, g32, label="H-α")
    fig[1].set_xlabel("absorption length (m)")
    fig[1].set_ylabel("escape factor")
    fig[1].set_title("escape factors")
    fig[1].legend()

Lyman-α is trapped first (a resonance line of the ground state); H-α only once n=2 is populated enough. The emission uses the trapped rates: with n=2 filled by the trapped Lyman-α, H-α is partly trapped itself, so its emissivity grows less than the n=3 population:

In [ ]:
thick = model.solve_steady_state(base.replace(trapping=opencr.RadiationTrapping(1.0)))
for result, label in ((thin, "thin"), (thick, "L = 1 m")):
    lines = result.emission()
    print(
        f"{label:7s} n(3) = {result.population('H(n=3)'):.3e} m^-3, "
        f"H-α = {lines['emissivity'].sel(line='A:3-2').item():.3e} photons m^-3 s^-1"
    )

A factor given in `Conditions.escape` overrides the computed one for that line, for example to keep a line optically thin in a test:

In [ ]:
forced = model.solve_steady_state(
    base.replace(trapping=opencr.RadiationTrapping(1.0), escape={"A:2-1": 1.0})
)
print(
    {
        line: round(forced.conditions.escape[line], 4)
        for line in ("A:2-1", "A:3-1", "A:3-2")
    }
)